In [ ]:
from mindspore_xai.explainer import LIMETabular

In [6]:
import sklearn.datasets
import sklearn.model_selection
import mindspore as ms
from mindspore.nn import SoftmaxCrossEntropyWithLogits
from mindspore import Model
from mindspore import dataset
iris = sklearn.datasets.load_iris()
train, test, labels_train, labels_test = sklearn.model_selection.train_test_split(iris.data, iris.target,
                                                                                      random_state=0,
                                                                                      train_size=0.80)

feature_names = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
class_names = ['setosa', 'versicolor', 'virginica']

In [ ]:
import mindspore.nn as nn

class LinearNet(nn.Cell):
    def __init__(self, num_inputs, num_class):
        super(LinearNet, self).__init__()
        self.linear = nn.Dense(num_inputs, num_class, activation=nn.Softmax())

    def construct(self, x):
        x = self.linear(x)
        return x
    
net = LinearNet(4, 3)

In [ ]:
train = ms.Tensor(train, ms.float32)
test = ms.Tensor(test, ms.float32)
labels_train = ms.Tensor(labels_train, ms.int32)
labels_test = ms.Tensor(labels_test, ms.int32)

XY_train = list(zip(train, labels_train))
ds_train = dataset.GeneratorDataset(XY_train, ['x', 'y'])
ds_train = ds_train.shuffle(buffer_size=120).batch(32, drop_remainder=True)

In [ ]:
net_loss = SoftmaxCrossEntropyWithLogits(True, reduction='mean')
opt = nn.Momentum(net.trainable_params(), learning_rate=0.005, momentum=0.9)
model = Model(net, net_loss, opt)
epoch = 500
model.train(epoch, ds_train, dataset_sink_mode=False)

In [ ]:
lime = LIMETabular(net, train, feature_names=feature_names, class_names=class_names)

print('train: ', type(train), train.shape)
print('number of features: ', len(feature_names))
print('feature_names: ', feature_names)
print('number of class: ', len(class_names))
print('class_names: ', class_names)

In [ ]:
inputs = test[:2]
targets = ms.Tensor([[0, 2], [1, 2]])
# For the first input, explain class 0 and class 2
# For the second input, explain class 1 and class 2
outputs = lime(inputs, targets)
# outputs format: [[description, weight]]

print('inputs: ', type(inputs), inputs.shape)
print('targets: ', type(targets), targets.shape)
print('outputs: ', type(outputs))

In [ ]:
for i, exps in enumerate(outputs):
    for j, exp in enumerate(exps):
        print('Local explanation for sample {} class {}'.format(i, class_names[targets[i][j]]))
        print(exp, '\n')